# 01 - Simple Prompt Chain

## Learning objective

Build and inspect a simple fixed LangChain workflow:

```text
Input data -> PromptTemplate -> formatted prompt -> Gemini -> AIMessage -> readable text
```

This notebook separates each stage into a small cell so that the syntax and intermediate objects can be inspected independently.

> **API quota note:** Only the cell labeled **Call Gemini** makes an API request. Re-running the inspection cells does not consume Gemini quota.


## 1. Check the active Python kernel

A Jupyter notebook executes code through a **kernel**. The executable below should point to the intended Conda environment, such as `nlp_env`.


In [1]:
import sys

print("Python executable:", sys.executable)
print("Python version:", sys.version)


Python executable: C:\Users\inkk1r04\AppData\Local\anaconda3\envs\nlp_env\python.exe
Python version: 3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]


## 2. Optional package installation

The notebook folder and Python-script folder are siblings. If the course packages are not installed in the active kernel, remove the `#` from the next line and run the cell once.

After installation, use **Kernel -> Restart Kernel** in the Jupyter menu.


In [ ]:
# !pip install -r "../langchain-course/requirements.txt"


## 3. Corporate SSL setup

`truststore` makes Python use the Windows certificate store. Run this before importing libraries that create HTTPS clients.


In [2]:
import truststore

truststore.inject_into_ssl()

print("Windows trust store injected successfully.")


Windows trust store injected successfully.


## 4. Load environment variables

The existing `.env` remains in the sibling `langchain-course` folder. This avoids maintaining a second copy of API keys.

The code checks whether variables exist but does **not** print their secret values.


In [3]:
import os
from pathlib import Path

from dotenv import load_dotenv

ENV_PATH = Path("../langchain-course/.env").resolve()

print("Expected .env path:", ENV_PATH)
print(".env file exists:", ENV_PATH.exists())

load_dotenv(dotenv_path=ENV_PATH)

print("Google API key loaded:", bool(os.getenv("GOOGLE_API_KEY")))
print("LangSmith API key loaded:", bool(os.getenv("LANGSMITH_API_KEY")))
print("LangSmith tracing enabled:", os.getenv("LANGSMITH_TRACING"))


Expected .env path: C:\Users\inkk1r04\OneDrive - Kellogg Company\Desktop\Work\0. DS Capability\Courses\Langchain\langchain-course\.env
.env file exists: True
Google API key loaded: True
LangSmith API key loaded: True
LangSmith tracing enabled: true


## 5. Import LangChain components

- `PromptTemplate` creates reusable prompt text with named placeholders.
- `ChatGoogleGenerativeAI` is LangChain's Gemini chat-model integration.


In [4]:
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

print("LangChain imports completed successfully.")


LangChain imports completed successfully.


## 6. Create the input information

This is the source information supplied to the model. The prompt will instruct Gemini not to add facts absent from this text.


In [5]:
information = """
Northstar Analytics is a fictional data-products company founded in 2022.
The company builds demand-forecasting and pricing applications for retail teams.
Its first product reduced the time required to prepare a weekly pricing report
from six hours to forty-five minutes. The company uses Python for analytics,
Power BI for business reporting, and cloud services for application deployment.
In 2025, Northstar Analytics started an internal learning program to help
analysts understand machine learning, data engineering, and generative AI.
"""

print(information)



Northstar Analytics is a fictional data-products company founded in 2022.
The company builds demand-forecasting and pricing applications for retail teams.
Its first product reduced the time required to prepare a weekly pricing report
from six hours to forty-five minutes. The company uses Python for analytics,
Power BI for business reporting, and cloud services for application deployment.
In 2025, Northstar Analytics started an internal learning program to help
analysts understand machine learning, data engineering, and generative AI.



## 7. Define the prompt template text

`{information}` is a placeholder. LangChain replaces the placeholder with the value supplied during formatting or invocation.


In [6]:
summary_template = """
Given only the following information:

{information}

Create:
1. A short summary
2. Two interesting facts explicitly stated in the information

Do not add facts, dates, organizations, or events that are not present
in the supplied information.
"""

print(summary_template)



Given only the following information:

{information}

Create:
1. A short summary
2. Two interesting facts explicitly stated in the information

Do not add facts, dates, organizations, or events that are not present
in the supplied information.



## 8. Create the `PromptTemplate` object

`input_variables=["information"]` tells LangChain which value the template expects.


In [7]:
summary_prompt = PromptTemplate(
    input_variables=["information"],
    template=summary_template,
)

print("Prompt object type:", type(summary_prompt))
print("Expected variables:", summary_prompt.input_variables)


Prompt object type: <class 'langchain_core.prompts.prompt.PromptTemplate'>
Expected variables: ['information']


In [8]:
summary_prompt

PromptTemplate(input_variables=['information'], input_types={}, partial_variables={}, template='\nGiven only the following information:\n\n{information}\n\nCreate:\n1. A short summary\n2. Two interesting facts explicitly stated in the information\n\nDo not add facts, dates, organizations, or events that are not present\nin the supplied information.\n')

## 9. Format and inspect the final prompt

This step does **not** call Gemini. It shows the exact text that will be sent to the model.


In [9]:
formatted_prompt = summary_prompt.format(
    information=information
)

print(formatted_prompt)



Given only the following information:


Northstar Analytics is a fictional data-products company founded in 2022.
The company builds demand-forecasting and pricing applications for retail teams.
Its first product reduced the time required to prepare a weekly pricing report
from six hours to forty-five minutes. The company uses Python for analytics,
Power BI for business reporting, and cloud services for application deployment.
In 2025, Northstar Analytics started an internal learning program to help
analysts understand machine learning, data engineering, and generative AI.


Create:
1. A short summary
2. Two interesting facts explicitly stated in the information

Do not add facts, dates, organizations, or events that are not present
in the supplied information.



## 10. Initialize Gemini

Creating the model object does not normally send a model request. The model name is stored in `MODEL` so it can be changed in one place.

If the model is unavailable or its free quota is exhausted, replace the value with another eligible model shown in Google AI Studio.


In [10]:
MODEL = "gemini-3.6-flash"

llm = ChatGoogleGenerativeAI(
    model=MODEL,
    max_retries=2,
)

print("Model object type:", type(llm))
print("Configured model:", MODEL)


Model object type: <class 'langchain_google_genai.chat_models.ChatGoogleGenerativeAI'>
Configured model: gemini-3.6-flash


## 11. Compose the chain

The pipe operator `|` connects components:

```text
PromptTemplate output -> Gemini input
```

This is a **fixed chain**, not an agent. The model does not choose tools or repeat steps.


In [12]:
chain = summary_prompt | llm

print("Chain object type:", type(chain))
print(chain)


Chain object type: <class 'langchain_core.runnables.base.RunnableSequence'>
first=PromptTemplate(input_variables=['information'], input_types={}, partial_variables={}, template='\nGiven only the following information:\n\n{information}\n\nCreate:\n1. A short summary\n2. Two interesting facts explicitly stated in the information\n\nDo not add facts, dates, organizations, or events that are not present\nin the supplied information.\n') middle=[] last=ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.6.4', 'langchain': '1.4.2', 'langchain-google-genai': '4.4.0'}}, profile={'name': 'Gemini 3.6 Flash', 'release_date': '2026-07-21', 'last_updated': '2026-07-21', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_cal

## 12. Call Gemini

> **This cell makes a real Gemini API request and may create a LangSmith trace.**

Run this cell only when API access and quota are available.


In [13]:
response = chain.invoke(
    {"information": information}
)

print("Gemini call completed.")


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Gemini call completed.


## 13. Inspect the response

`response` is an `AIMessage`, not a plain string.

- `response.text` gives readable text.
- `response.content` may contain provider-specific structured blocks.
- `response.response_metadata` contains model and request metadata.


In [15]:
print("Response object type:", type(response))
print("Readable response:")
print(response.text)


Response object type: <class 'langchain_core.messages.ai.AIMessage'>
Readable response:
**Summary**
Founded in 2022, Northstar Analytics is a fictional data-products company that builds demand-forecasting and pricing applications for retail teams using Python, Power BI, and cloud services. Its initial product dramatically reduced weekly pricing report preparation time, and in 2025, the company launched an internal learning program to educate its analysts in machine learning, data engineering, and generative AI.

**Two Interesting Facts**
1. The company's very first product reduced the time needed to prepare a weekly pricing report from six hours to forty-five minutes.
2. In 2025, Northstar Analytics began an internal learning program to teach its analysts machine learning, data engineering, and generative AI.


In [17]:
print("Raw content:")
print(response.content)

print("Response metadata:")
print(response.response_metadata)


Raw content:
[{'type': 'text', 'text': "**Summary**\nFounded in 2022, Northstar Analytics is a fictional data-products company that builds demand-forecasting and pricing applications for retail teams using Python, Power BI, and cloud services. Its initial product dramatically reduced weekly pricing report preparation time, and in 2025, the company launched an internal learning program to educate its analysts in machine learning, data engineering, and generative AI.\n\n**Two Interesting Facts**\n1. The company's very first product reduced the time needed to prepare a weekly pricing report from six hours to forty-five minutes.\n2. In 2025, Northstar Analytics began an internal learning program to teach its analysts machine learning, data engineering, and generative AI.", 'extras': {'signature': 'Er4TCrsTAWkUfROpH8xJJWMOvmV9ufyiUI0QNO6obSP5sfkJ1eRF/Z3lzZpK1RrVwtIa9yH+kZXtXvebI3SWhe+u+biN+0Bu5DAGDU4RBrzlD2rur9QSVwT2BpoWfIS/F1STgARCHVs/+LW/CtpXgNfJ6jWK0sS0AvIChvU230xDxMdHIS/ef1W8ug0l4iVO3RX

## 14. Key takeaways

1. A `PromptTemplate` is reusable text with named placeholders.
2. `.format()` lets you inspect the completed prompt without calling the model.
3. The `|` operator composes a fixed sequence of components.
4. `.invoke()` executes the chain.
5. The returned object is an `AIMessage`.
6. `response.text` is usually the easiest way to print Gemini's readable answer.
7. A chain follows a predetermined path. An agent can choose tools and repeat steps.

## Practice exercises

- Add a third requested fact.
- Change the source information.
- Add a maximum word limit to the summary.
- Add another placeholder and inspect `summary_prompt.input_variables`.
- Format and inspect the prompt before making an API request.
